# 12 Catalyst and Tungsten

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Explain the two engines behind Spark SQL performance: <b>Catalyst</b>, the query optimizer (analysis, rule-based and cost-based optimization, physical planning), and <b>Tungsten</b>, the execution engine (binary memory format, whole-stage code generation, vectorized reading). See their effects in plans, and know where Databricks <b>Photon</b> fits in.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"Why are DataFrames faster than RDDs?" and "what is the Catalyst optimizer?" are classic interview questions. Knowing the optimizer's rules also tells you what you <b>don't</b> need to hand-optimize (filter order, column pruning, constant expressions) and what it <b>can't</b> fix (UDFs, bad data layout, skew).
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Catalyst: from query to plan</b>
<ol><li><b>Analysis</b>: resolve tables, columns and functions against the catalog, and check types</li><li><b>Logical optimization</b>: apply rules repeatedly until the plan stops changing. Examples:</li></ol>
<ul><li>Predicate pushdown and <b>combining filters</b></li><li><b>Column pruning</b></li><li><b>Constant folding</b> (<code>1 + 2</code> → <code>3</code>) and <b>boolean simplification</b> (<code>x AND true</code> → <code>x</code>)</li><li>Null propagation, eliminating unnecessary sorts, collapsing projections</li></ul>
<ol><li><b>Physical planning</b>: generate candidate physical plans and choose operators (join strategies, aggregation types), using size <b>statistics</b> where available (cost-based optimization, CBO)</li><li><b>Code generation</b>: hand the physical plan to Tungsten</li></ol>
Then <b>AQE</b> may re-optimize at runtime with real sizes (lesson 13).
</div>
<br>
<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Tungsten: making execution fast</b>
<ul><li><b>Binary row format (UnsafeRow)</b>: rows stored as compact bytes, outside Java object overhead and with less garbage-collection pressure</li><li><b>Whole-stage code generation</b>: a chain of operators is compiled into <b>one</b> Java function with a tight loop, instead of calling each operator per row (no virtual function calls, values kept in CPU registers)</li><li><b>Vectorized readers</b>: Parquet/ORC are read in columnar batches</li><li><b>Cache-aware algorithms</b>: sorting and hashing designed around CPU caches</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A team rewrote an RDD-based pipeline (Python lambdas over tuples) as DataFrame code with the same logic. Catalyst pushed their filters into the Parquet scan and pruned 40 of 50 columns. Tungsten generated tight code for the remaining expressions, with no Python involved. The job went from 3 hours to 12 minutes. On Databricks, enabling Photon made the same DataFrame code another 2× faster with no code change.
</div>

## 1. Constant folding and boolean simplification

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes deliberately redundant expressions and shows how Catalyst simplifies them in the optimized plan.<br><br>
<b>Why it matters</b><br>You can write readable code (for example <code>F.lit(60) * 60</code> for "seconds per hour") without worrying about runtime cost: the optimizer computes constants once at planning time and removes always-true conditions.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
In the <b>Optimized</b> plan: <code>3600</code> instead of <code>60 * 60</code>, the <code>1 = 1</code> condition removed, and <code>id &gt; 5</code> kept as the only filter.
</div>

In [0]:
from pyspark.sql import functions as F

(
    spark.range(100)
    .withColumn("seconds", F.col("id") * (F.lit(60) * F.lit(60)))
    .filter((F.lit(1) == F.lit(1)) & (F.col("id") > 5))
).explain(True)

## 2. Combining filters, collapsing projections, and null propagation

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Chains several <code>filter</code> and <code>withColumn</code> calls, plus an expression on a null literal, and inspects the optimized plan.<br><br>
<b>Why it matters</b><br>Many small steps in code become one filter and one projection in the plan. Expressions on a known null become <code>null</code> immediately. Write clear code in small steps: Catalyst merges them.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The optimized plan shows a single <code>Filter</code> combining all three conditions, a single <code>Project</code>, and <code>null</code> for the column built from <code>lit(None)</code>.
</div>

In [0]:
(
    spark.range(1000)
    .filter("id > 10")
    .withColumn("a", F.col("id") + 1)
    .filter("id < 900")
    .withColumn("b", F.col("a") * 2)
    .withColumn("nothing", F.lit(None).cast("int") + F.col("id"))
    .filter("id % 2 = 0")
).explain("extended")

## 3. Optimizations Catalyst can't do: UDFs are black boxes

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Filters with a built-in expression and with an equivalent Python UDF, on data written to Parquet, and compares where the filter ends up.<br><br>
<b>Why it matters</b><br>Catalyst knows the semantics of built-in functions, so it can push them into the scan. It knows nothing about a UDF, so the UDF filter can't be pushed down, can't be combined, and blocks code generation. This is why "avoid UDFs" is about the <b>optimizer</b> as much as about Python speed.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The built-in version shows the condition in <code>PushedFilters</code>. The UDF version shows an empty or <code>IsNotNull</code>-only <code>PushedFilters</code>, with a separate <code>BatchEvalPython</code> step and a <code>Filter</code> on top.
</div>

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.spark_basics")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.spark_basics.raw_files")
path = "/Volumes/workspace/spark_basics/raw_files/lesson02_12/numbers"

In [0]:
from pyspark.sql.functions import udf

spark.range(100_000).withColumn("amount", F.col("id") % 1000).write.mode("overwrite").parquet(path)
numbers = spark.read.parquet(path)

@udf("boolean")
def is_big(x):
    return x is not None and x > 900

print("---- built-in ----")
numbers.filter(F.col("amount") > 900).explain()
print("---- UDF ----")
numbers.filter(is_big("amount")).explain()

## 4. Whole-stage code generation

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints the generated code for a small pipeline with <code>explain("codegen")</code>.<br><br>
<b>Why it matters</b><br>This is Tungsten at work: the <code>Range</code>, <code>Filter</code> and <code>Project</code> are fused into one generated class with a single <code>processNext</code> loop. The <code>*(1)</code> prefixes in normal plans mark the operators that belong to one generated function.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
"Found 1 WholeStageCodegen subtrees", followed by generated Java code containing a <code>processNext()</code> method that loops over rows and applies the filter and the multiplication inline. Only the first lines are printed.
</div>

In [0]:
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    spark.range(1000).filter("id % 3 = 0").select((F.col("id") * 7).alias("x")).explain("codegen")
code = buf.getvalue().splitlines()
print("\n".join(code[:6]))
print("...")
print("\n".join(l for l in code if "processNext" in l or "while" in l)[:600])

## 5. Cost-based optimization uses statistics

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the size estimates Catalyst uses (<code>explain("cost")</code>), then computes column statistics for a Delta table with <code>ANALYZE TABLE</code> so the estimates become more precise.<br><br>
<b>Why it matters</b><br>Join strategy and join order depend on size estimates. Without statistics, Spark estimates from file sizes, which can be badly wrong after filters. <code>ANALYZE TABLE ... COMPUTE STATISTICS</code> gives the optimizer row counts and column distributions. On Databricks, Delta tables also keep file-level statistics automatically, and predictive optimization can maintain table statistics for you.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A plan annotated with <code>Statistics(sizeInBytes=...)</code>. After <code>ANALYZE</code> on Databricks, the statistics also include <code>rowCount</code>.
</div>

In [0]:
numbers.filter("amount = 7").explain("cost")

In [0]:
table = "workspace.spark_basics.lesson02_12_numbers"
numbers.write.mode("overwrite").saveAsTable(table)
spark.sql(f"ANALYZE TABLE {table} COMPUTE STATISTICS FOR ALL COLUMNS")
spark.table(table).filter("amount = 7").explain("cost")
spark.sql(f"DROP TABLE IF EXISTS {table}")

## 6. Why DataFrames beat RDDs, and where Photon fits

| | RDD (Python lambdas) | DataFrame / SQL | DataFrame + Photon (Databricks) |
|---|---|---|---|
| Optimizer | None: runs exactly as written | Catalyst | Catalyst |
| Data format | Python objects, pickled | Tungsten binary rows | Columnar, vectorized |
| Execution | Python per row | Generated JVM code | Native C++ vectorized engine |
| Serverless | ❌ Not available | ✅ | ✅ Photon is on by default |

**Photon** is Databricks' native vectorized query engine, written in C++. It replaces many Tungsten operators at execution time (you'll see `Photon...` operators in plans and query profiles) and speeds up SQL and DataFrame workloads without code changes. It doesn't run Python UDFs, so those still execute in Python.

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Serverless compute uses Photon. In a Free Edition query profile, look for operators whose names start with <code>Photon</code>.
</div>

## Under the hood

```
DataFrame / SQL
   │
   ▼  CATALYST
 Unresolved plan ─analysis─▶ Resolved plan ─rules─▶ Optimized logical plan ─strategies + stats─▶ Physical plan
                                     ▲                                                                 │
                                  catalog                                                              ▼
   ┌───────────────────────────────────────── TUNGSTEN ───────────────────────────────────────────────┘
   │ whole-stage codegen (one Java function per stage) · UnsafeRow binary format · vectorized Parquet reader
   ▼
 Executors run the generated code  (or Photon's native operators on Databricks)
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> optimization happens at the action, on the whole plan.

The plan below shows both: Catalyst moved the filter below the projection and pruned columns, and Tungsten fused everything into the `*(1)` codegen stage.

In [0]:
numbers.select("id", "amount", (F.col("amount") * 2).alias("double")).filter("amount > 10").explain()

In [0]:
# Optional cleanup
dbutils.fs.rm("/Volumes/workspace/spark_basics/raw_files/lesson02_12", True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: hand-reordering filters and selects for performance</b><br>
Catalyst already pushes filters down and prunes columns. Focus on what it can't fix: UDFs, data layout, skew.<br><br>
<b>⛔ Problem: a filter isn't pushed into the scan</b><br>It uses a UDF or a non-deterministic expression. Rewrite it with built-in functions.<br><br>
<b>⛔ Problem: wrong join strategy because of bad size estimates</b><br>Compute statistics (<code>ANALYZE TABLE</code>), add a hint, or rely on AQE.<br><br>
**⛔ Problem: no <code>*(n)</code> codegen prefix on some operators**<br>Codegen stops at Python UDFs and some complex operators. Fewer UDFs means longer fused stages.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the Catalyst optimizer?</b><br>
Spark SQL's extensible query optimizer. It analyzes a query against the catalog, applies rule-based optimizations such as predicate pushdown, column pruning, constant folding and combining filters, uses statistics for cost-based choices like join strategy, and produces a physical plan.<br><br>

<b>🎤 2. What are the phases of Catalyst?</b><br>
Analysis (resolve names and types), logical optimization (rule-based rewrites), physical planning (choose operators, with cost-based decisions), and code generation for execution.<br><br>

<b>🎤 3. What is Tungsten?</b><br>
Spark's execution engine work focused on CPU and memory efficiency: a compact binary row format, off-heap memory management, whole-stage code generation that fuses operators into a single function, and vectorized columnar reads.<br><br>

<b>🎤 4. What is whole-stage code generation?</b><br>
Spark compiles a chain of operators within a stage into one generated Java function with a tight loop, avoiding per-row virtual calls between operators. In plans, operators in the same generated function share a <code>*(n)</code> prefix.<br><br>

<b>🎤 5. Why are DataFrames faster than RDDs?</b><br>
DataFrames have a schema, so Catalyst can optimize the whole query and Tungsten can store rows in a compact binary format and generate efficient code. RDD operations are opaque functions over objects, executed as written, with object serialization overhead.<br><br>

<b>🎤 6. Why can't Catalyst optimize UDFs?</b><br>
A UDF is a black box: Catalyst doesn't know its logic, so it can't push it down, fold it, combine it or include it in code generation.<br><br>

<b>🎤 7. What is Photon?</b><br>
Databricks' native, vectorized query engine written in C++. It executes SQL and DataFrame operators faster than the JVM engine without code changes, and it's used by serverless compute and SQL warehouses.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which component of Spark rewrites a query by pushing filters down and pruning unused columns?</b><br>
A. The cluster manager<br>
B. The Catalyst optimizer<br>
C. The DAG scheduler<br>
D. The block manager<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. What is Photon on Databricks?</b><br>
A. A file format for Delta tables<br>
B. A native vectorized query engine that accelerates SQL and DataFrame workloads<br>
C. A library for Python UDFs<br>
D. The Unity Catalog permission system<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a query that multiplies <code>id</code> by <code>(24 * 60 * 60)</code> and filters on <code>id &gt; 0 AND true</code>. Confirm in the optimized plan that both are simplified</li><li>Rewrite <code>is_big</code> as a built-in expression and confirm the filter appears in <code>PushedFilters</code></li><li>Count the whole-stage codegen subtrees with <code>explain("codegen")</code> for a query <b>without</b> a shuffle and for one <b>with</b> a <code>groupBy</code>. Explain the result for the second one</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
import re

# 1. Constant folding and boolean simplification
spark.range(10).withColumn("secs", F.col("id") * (F.lit(24) * 60 * 60)).filter((F.col("id") > 0) & F.lit(True)).explain(True)

# 2. Built-in filter is pushed down
numbers.filter(F.col("amount") > 900).explain()

# 3. Count codegen subtrees in the plan
def codegen_subtrees(df):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain("codegen")
    return re.search(r"Found (\d+) WholeStageCodegen", buf.getvalue()).group(1)

print("no shuffle  :", codegen_subtrees(numbers.filter("amount > 10").select("id")))
print("with groupBy:", codegen_subtrees(numbers.groupBy("amount").count()))
# With AQE, a query with a shuffle is planned stage by stage at runtime, so the initial plan has
# no codegen yet (0 subtrees). The final plan in the Spark UI / query profile shows one per stage.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Catalyst: analysis → rule-based logical optimization → physical planning (with stats) → codegen</li><li>Rules you get for free: pushdown, pruning, constant folding, combining filters, collapsing projects, removing useless sorts</li><li>Catalyst can't see inside UDFs: no pushdown, no folding, no codegen</li><li>Tungsten: binary rows, whole-stage codegen (<code>*(n)</code>, visible in plans without a shuffle, or in the final AQE plan), vectorized reads</li><li>Statistics (<code>ANALYZE TABLE</code>) improve size estimates and join choices</li><li>Photon (Databricks) is a native vectorized engine, on by default for serverless</li></ul>
</div>

| See it | How |
|---|---|
| Optimizer rewrites | `df.explain(True)` |
| Generated code | `df.explain("codegen")` |
| Size estimates | `df.explain("cost")` |
| Table statistics | `ANALYZE TABLE t COMPUTE STATISTICS FOR ALL COLUMNS` |
| Photon | Operator names in the query profile |

## Git commit

```
git add 02_spark_internals/12_catalyst_and_tungsten.ipynb
git commit -m "add 02_12 catalyst and tungsten notebook"
```